# Beca 18 RAG — el proyecto completo en 10 celdas**Data Science con Python · Universidad del Pacífico · 2026-II**Jefe de práctica: Paul Melo Ramos · Profesor: Alexander QuispeRepositorio: `github.com/patrickmelorr-lang/BECA-18-RAG`---Este notebook es **autocontenido**: descarga el documento, construye el índice, llama a la API yevalúa el sistema. No necesitas clonar nada. Corre en Google Colab y en local.Documento: **RDE N.° 033-2026-MINEDU/VMGI-PRONABEC**, Reglamento de Beca 18, 138 páginas.## El pipeline```OFFLINE (celdas 3 a 6)  PDF ──▶ páginas ──▶ fragmentos ──▶ vectores ──▶ ChromaDBONLINE (celdas 7 a 10)  pregunta ──▶ vector ──▶ top-k coseno ──▶ ¿supera el umbral?                                             ├─ no ──▶ "no sé"  (sin gastar una llamada)                                             └─ sí ──▶ prompt aumentado ──▶ DeepSeek                                                          └──▶ respuesta + página + costo```## Las 10 celdas| # | Qué hace | El número que sale ||---|---|---|| 1 | esta portada | — || 2 | instalar, descargar el PDF, cargar la llave | 138 páginas || 3 | extraer el texto y **demostrar el bug de la cita de página** | 9.3% contra 100% || 4 | **chunking en vivo**: tamaño, solape, comparación | 1,483 contra 570 fragmentos || 5 | embeddings locales y la letra chica del modelo | orden con y sin prefijos || 6 | indexar en ChromaDB | segundos, no minutos || 7 | búsqueda semántica y su fragilidad | similitud y puesto || 8 | generación con DeepSeek: umbral, prompt, costo | USD por consulta || 9 | evaluación: Recall@k y abstención | 0.33 / 0.67 / 0.83 || 10 | batería de preguntas y cierre | costo total de la sesión |> **Necesitas una llave de DeepSeek** (`platform.deepseek.com`) solo a partir de la celda 8.> Las celdas 1 a 7 corren gratis, porque los embeddings son locales.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════#  CELDA 2 — SETUP: dependencias, documento y llave# ═══════════════════════════════════════════════════════════════════════════import importlib.util, subprocess, sys, os, re, time, csv, json, urllib.requestfrom pathlib import Path# --- 1. Instalar solo lo que falte -----------------------------------------REQ = [("pypdf", "pypdf"),       ("langchain-text-splitters", "langchain_text_splitters"),       ("chromadb", "chromadb"),       ("sentence-transformers", "sentence_transformers"),       ("openai", "openai")]faltan = [p for p, m in REQ if importlib.util.find_spec(m) is None]if faltan:    print("Instalando:", ", ".join(faltan))    rc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan]).returncode    if rc != 0:        # Algunos Linux y macOS bloquean pip fuera de un entorno virtual (PEP 668)        rc = subprocess.run([sys.executable, "-m", "pip", "install", "-q",                             "--break-system-packages", *faltan]).returncode    if rc != 0:        print("\nNo pude instalar solo. Corre esto en una terminal y reinicia el kernel:")        print("  python -m venv .venv")        print("  .venv\\Scripts\\activate      (Windows)")        print("  source .venv/bin/activate     (macOS / Linux)")        print("  pip install " + " ".join(faltan))        raise SystemExit("Instalacion pendiente")import numpy as np# --- 2. Configuración: TODOS los parámetros viven aquí ----------------------# Regla del proyecto: si un número aparece más abajo, está mal. Va en este dict.CONFIG = {    "pdf_url": "https://raw.githubusercontent.com/patrickmelorr-lang/"               "BECA-18-RAG/main/data/beca18_reglamento.pdf",    "pdf_local": "beca18_reglamento.pdf",    "documento": "RDE N.o 033-2026-MINEDU/VMGI-PRONABEC",    "chunk_tamano": 900,    "chunk_solape": 150,    "separadores": ["\n\n", "\n", ". ", " "],    "modelo_emb": "intfloat/multilingual-e5-small",    "prefijo_doc": "passage: ",    "prefijo_query": "query: ",    "lote": 64,    "chroma_path": "chroma_beca18",    "coleccion": "beca18",    "base_url": "https://api.deepseek.com",    "modelo_llm": "deepseek-flash",    "temperature": 0.1,    "max_tokens": 500,    "k": 5,    "umbral_similitud": 0.78,    # Precios SIEMPRE con fecha y fuente. Un precio sin fecha no es auditable.    "precios": {        "verificado_el": "2026-09-14",        "fuente": "https://api-docs.deepseek.com/quick_start/pricing",        "in": 0.15, "in_cache": 0.003, "out": 0.60,   # USD por millón de tokens    },}# --- 3. Descargar el documento ---------------------------------------------if not Path(CONFIG["pdf_local"]).exists():    print("Descargando el reglamento desde el repositorio...")    urllib.request.urlretrieve(CONFIG["pdf_url"], CONFIG["pdf_local"])print(f"PDF listo: {Path(CONFIG['pdf_local']).stat().st_size/1e6:.1f} MB")# --- 4. La llave: nunca escrita dentro del código ---------------------------API_KEY = Nonetry:    from google.colab import userdata          # Colab: panel de Secrets    API_KEY = userdata.get("DEEPSEEK_API_KEY")except Exception:    API_KEY = os.getenv("DEEPSEEK_API_KEY")    # local: variable de entorno o .envif not API_KEY:    from getpass import getpass    print("\nNo encontré la llave. Pégala aquí (no se muestra al escribir).")    print("La necesitas recién en la celda 8; puedes dejarla vacía por ahora.")    API_KEY = getpass("DEEPSEEK_API_KEY: ").strip() or Noneprint("Llave cargada:", bool(API_KEY))print("\nSetup completo. La configuración vive en CONFIG, no en el código.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════#  CELDA 3 — EXTRACCIÓN, y el bug de la cita de página##  El system prompt le ordenará al modelo: "cita siempre la página".#  La primera versión de este proyecto insertaba el marcador [PAGE 12] DENTRO#  del texto y después partía todo junto. Vamos a medir qué pasó con eso.# ═══════════════════════════════════════════════════════════════════════════import pypdffrom langchain_text_splitters import RecursiveCharacterTextSplitterdef limpiar(t):    t = re.sub(r" {2,}", " ", t)                 # espacios múltiples    t = re.sub(r"(?<!\n)\n(?!\n)", " ", t)       # saltos de línea sueltos    return t.strip()def extraer_paginas(ruta, minimo=30):    """PDF -> [{'pagina': n, 'texto': str}].  UNA ENTRADA POR PÁGINA.    La clave está en NO concatenar: si troceamos página por página, el número    de página puede viajar como METADATO en vez de como texto.    """    lector = pypdf.PdfReader(ruta)    paginas, vacias = [], 0    for n, p in enumerate(lector.pages, 1):        t = limpiar(p.extract_text() or "")        if len(t) < minimo:            vacias += 1            continue        paginas.append({"pagina": n, "texto": t})    return paginas, vaciaspaginas, vacias = extraer_paginas(CONFIG["pdf_local"])chars = sum(len(p["texto"]) for p in paginas)palabras = sum(len(p["texto"].split()) for p in paginas)tokens_est = round(chars / 3)          # español: ~3 caracteres por tokenprint("EL DOCUMENTO")print(f"  páginas útiles   : {len(paginas)}  (descartadas por vacías: {vacias})")print(f"  caracteres       : {chars:,}")print(f"  palabras         : {palabras:,}")print(f"  tokens estimados : {tokens_est:,}")print(f"  tokens/palabra   : {tokens_est/palabras:.2f}   (en inglés lo típico es 1.3)")# ---------------------------------------------------------------------------#  ENFOQUE VIEJO: el marcador [PAGE N] como texto, y después partir todo junto# ---------------------------------------------------------------------------texto_pegado = "\n\n".join(f"[PAGE {p['pagina']}]\n{p['texto']}" for p in paginas)sp_viejo = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60,                                          separators=CONFIG["separadores"])chunks_viejo = sp_viejo.split_text(texto_pegado)con_marcador = sum(1 for c in chunks_viejo if "[PAGE" in c)print("\nENFOQUE VIEJO — marcador dentro del texto")print(f"  fragmentos                : {len(chunks_viejo):,}")print(f"  pueden citar su página    : {con_marcador:,}  "      f"({con_marcador/len(chunks_viejo)*100:.1f}%)")print(f"  NO pueden citar su página : {len(chunks_viejo)-con_marcador:,}  "      f"({(len(chunks_viejo)-con_marcador)/len(chunks_viejo)*100:.1f}%)")print("""  Qué pasa en producción: recuperas 5 fragmentos, uno solo trae marcador,  el modelo obedece la orden de citar y le atribuye TODO a esa página.  No es una cita faltante: es una cita confiadamente EQUIVOCADA.  El modelo no mintió. Hizo lo más probable con lo que le dimos.  El error fue de arquitectura.  Arreglo:  TEXTO en el texto.  DATOS SOBRE el texto, en METADATA.""")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════#  CELDA 4 — CHUNKING EN VIVO##  Trocear = cortar el documento en tiritas para poder buscar rápido.#  Dos perillas:  TAMAÑO (cuánto mide cada tirita)#                 SOLAPE (cuántas letras repite de la tirita anterior)# ═══════════════════════════════════════════════════════════════════════════def trocear(paginas, tamano, solape, separadores):    """Trocea PÁGINA POR PÁGINA y le cuelga el número como metadato."""    sp = RecursiveCharacterTextSplitter(chunk_size=tamano, chunk_overlap=solape,                                        separators=separadores)    out = []    for p in paginas:        for pedazo in sp.split_text(p["texto"]):            out.append({"id": f"c{len(out):05d}", "texto": pedazo,                        "pagina": p["pagina"]})    return out# --- A. ¿Por qué existe el solape? Se ve mejor que se explica ---------------demo = [{"pagina": 1, "texto": paginas[30]["texto"][:700]}]sin_solape = trocear(demo, 300, 0,   CONFIG["separadores"])con_solape = trocear(demo, 300, 100, CONFIG["separadores"])print("SIN SOLAPE — la frase se parte y ninguna tirita se entiende sola")print(f"  tirita 1 termina en : ...{sin_solape[0]['texto'][-70:]!r}")print(f"  tirita 2 empieza en : {sin_solape[1]['texto'][:70]!r}...")print("\nCON SOLAPE 100 — la tirita 2 repite el final de la 1")print(f"  tirita 1 termina en : ...{con_solape[0]['texto'][-70:]!r}")print(f"  tirita 2 empieza en : {con_solape[1]['texto'][:70]!r}...")# --- B. El tamaño cambia todo ----------------------------------------------print(f"\n{'tamaño':>7} {'solape':>7} {'fragmentos':>12} {'largo prom.':>12}")print("-" * 42)for tam, ov in [(400, 60), (600, 100), (900, 150), (1200, 180)]:    ch = trocear(paginas, tam, ov, CONFIG["separadores"])    prom = round(sum(len(c["texto"]) for c in ch) / len(ch))    marca = "  <-- el que usamos" if tam == CONFIG["chunk_tamano"] else ""    print(f"{tam:>7} {ov:>7} {len(ch):>12,} {prom:>12}{marca}")# --- C. El chunking definitivo, y la comprobación del arreglo --------------chunks = trocear(paginas, CONFIG["chunk_tamano"], CONFIG["chunk_solape"],                 CONFIG["separadores"])con_pag = sum(1 for c in chunks if c["pagina"])print(f"\nENFOQUE NUEVO — página como metadata")print(f"  fragmentos             : {len(chunks):,}")print(f"  pueden citar su página : {con_pag:,}  ({con_pag/len(chunks)*100:.1f}%)")print(f"  páginas cubiertas      : {len({c['pagina'] for c in chunks})}")print(f"\n  ANTES: {con_marcador/len(chunks_viejo)*100:.1f}%   "      f"AHORA: {con_pag/len(chunks)*100:.1f}%")c = chunks[100]print(f"\nEjemplo -> id={c['id']}  pagina={c['pagina']}  largo={len(c['texto'])}")print(f"  {c['texto'][:200]}...")print("""  ¿Por qué 900 y no 400?  Un artículo de un reglamento peruano suele pasar los  400 caracteres, así que ese tamaño parte artículos por la mitad.  La pista que delató el problema en la versión anterior: el control de k  estaba puesto en 13 aunque el valor por defecto era 5. Alguien subió k hasta  que la respuesta apareciera, en vez de preguntarse por qué no aparecía.  Subir k para tapar una recuperación mala es fuerza bruta: cuesta tokens y  mete ruido. Lo correcto es medirlo, y eso lo hacemos en la celda 9.""")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════#  CELDA 5 — EMBEDDINGS LOCALES  (sin API, sin costo, sin límites)##  Un token ID es un DNI: identifica pero no describe.#  Un EMBEDDING es la ficha completa: 384 números donde cada posición captura#  algo del significado. Frases parecidas dan listas de números parecidas.##  La primera versión usaba embeddings por API: 1,483 fragmentos en lotes de#  50 con una espera de 30 s entre lotes = MÁS DE 15 MINUTOS solo de esperas.#  Locales: decenas de segundos y costo cero.##  Lo que de verdad cambia: con 15 min por reindexado nadie prueba tres#  estrategias de chunking. Con 30 segundos, prueba diez.# ═══════════════════════════════════════════════════════════════════════════from sentence_transformers import SentenceTransformerprint(f"Cargando {CONFIG['modelo_emb']} (la primera vez descarga ~470 MB)...")modelo_emb = SentenceTransformer(CONFIG["modelo_emb"])DIM = modelo_emb.get_sentence_embedding_dimension()print(f"Dimensiones: {DIM}")def vec_documentos(textos):    """Vectores para los fragmentos del corpus."""    return modelo_emb.encode([CONFIG["prefijo_doc"] + t for t in textos],                             batch_size=CONFIG["lote"],                             normalize_embeddings=True,                             show_progress_bar=len(textos) > 200).tolist()def vec_consulta(texto):    """Vector para la pregunta del usuario."""    return modelo_emb.encode([CONFIG["prefijo_query"] + texto],                             normalize_embeddings=True)[0].tolist()def coseno(a, b):    a, b = np.asarray(a), np.asarray(b)    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))# --- La letra chica: los prefijos -------------------------------------------# multilingual-e5 fue entrenado exigiendo "query: " y "passage: ".# Omitirlos NO da ningún error. Solo empeora la recuperación, en silencio.pregunta = "¿Cuánto dinero me dan al mes?"candidatos = [    "La subvención de manutención se otorga mensualmente al becario.",    "El becario debe matricularse en el número mínimo de créditos.",    "La receta del ceviche lleva limón, cebolla y ají limo.",]vq, vd = vec_consulta(pregunta), vec_documentos(candidatos)crudo = modelo_emb.encode([pregunta] + candidatos, normalize_embeddings=True)print(f"\nPregunta: {pregunta}\n")print(f"{'candidato':<58}{'con prefijo':>13}{'sin prefijo':>13}")print("-" * 84)for i, c in enumerate(candidatos):    print(f"{c[:56]:<58}{coseno(vq, vd[i]):>13.4f}{coseno(crudo[0], crudo[i+1]):>13.4f}")v = np.array(vd[0])print(f"\nNorma del vector: {np.linalg.norm(v):.6f}  (debe ser 1.0)")print("""  Vectores normalizados => producto punto = similitud coseno.  Una multiplicación de matrices en vez de una división por normas: por eso  buscar entre un millón de vectores toma milisegundos.  AVISO: todas las similitudes de este modelo viven entre ~0.75 y ~0.90.  Un umbral como "mayor a 0.8 es relevante" NO se traslada entre modelos.  Se calibra con datos propios. Por eso vive en CONFIG.""")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════#  CELDA 6 — ÍNDICE VECTORIAL  (ChromaDB)##  Es el índice del libro: guardas cada tirita junto a su lista de números.#  Cuando preguntas algo, conviertes la pregunta en números y buscas las#  tiritas cuyos números apuntan en la misma dirección.# ═══════════════════════════════════════════════════════════════════════════import chromadbcliente = chromadb.PersistentClient(path=CONFIG["chroma_path"])col = cliente.get_or_create_collection(name=CONFIG["coleccion"],                                       metadata={"hnsw:space": "cosine"})ya = col.count()if ya < len(chunks):    if ya:        print(f"Reanudando desde el fragmento {ya}...")    t0 = time.time()    for i in range(ya, len(chunks), CONFIG["lote"]):        bloque = chunks[i:i + CONFIG["lote"]]        textos = [c["texto"] for c in bloque]        col.add(ids=[c["id"] for c in bloque],                documents=textos,                embeddings=vec_documentos(textos),                metadatas=[{"pagina": c["pagina"]} for c in bloque])    print(f"Indexado en {time.time()-t0:.1f} segundos "          f"(la versión por API tardaba más de 15 minutos solo en esperas)")else:    print("El índice ya estaba completo: no se reindexa nada. Eso es idempotencia.")print(f"Fragmentos en el índice: {col.count()}")def buscar(pregunta, k=None):    """Top-k por similitud coseno, con página y score."""    k = k or CONFIG["k"]    r = col.query(query_embeddings=[vec_consulta(pregunta)], n_results=k,                  include=["documents", "metadatas", "distances"])    return [{"texto": r["documents"][0][i],             "pagina": r["metadatas"][0][i]["pagina"],             "distancia": round(float(r["distances"][0][i]), 4),             "similitud": round(1 - float(r["distances"][0][i]), 4)}            for i in range(len(r["documents"][0]))]print("\nLA TRAMPA QUE ATRAPA A TODO EL MUNDO:")print("  Chroma devuelve DISTANCIA, no similitud.  similitud = 1 - distancia")print("  Un '0.41' en pantalla no es 41% de parecido: es 59%.\n")for i, f in enumerate(buscar("¿Cuánto es la subvención mensual?", 3), 1):    print(f"  {i}. página {f['pagina']:>3} | distancia {f['distancia']:.4f} "          f"| similitud {f['similitud']:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════#  CELDA 7 — BÚSQUEDA SEMÁNTICA: lo que puede y lo que no# ═══════════════════════════════════════════════════════════════════════════# --- A. El momento en que se entiende para qué sirve todo esto -------------# Preguntamos usando palabras que NO están en el documento.print("A) VOCABULARIO DISTINTO AL DEL DOCUMENTO\n")for q in ["¿Me devuelven la plata que gasté?",      # el doc dice "reembolso"          "¿Qué pasa si jalo muchos cursos?",       # el doc dice "desaprobar créditos"          "¿Me puedo cambiar de universidad?"]:     # el doc dice "traslado"    top = buscar(q, 2)    print(f"  {q}")    for f in top:        print(f"     pág {f['pagina']:>3} | sim {f['similitud']:.4f} | {f['texto'][:88]}...")    print()print("""  Cero palabras en común y aun así encuentra la zona correcta.  Ctrl+F habría devuelto NADA. Esa es la diferencia entre buscar por palabras  y buscar por significado.""")# --- B. Pero es frágil ------------------------------------------------------print("B) LA MISMA PREGUNTA, TRES FORMULACIONES\n")variantes = [    "¿Cuáles son las obligaciones del becario?",    # vocabulario del documento    "¿Qué tengo que cumplir si me gano la beca?",   # paráfrasis    "¿Qué me toca hacer como becario?",             # coloquial]objetivo = buscar(variantes[0], 10)[0]["pagina"]for q in variantes:    top10 = [f["pagina"] for f in buscar(q, 10)]    puesto = top10.index(objetivo) + 1 if objetivo in top10 else "fuera del top-10"    print(f"  {q}")    print(f"     top-10 páginas: {top10}")    print(f"     la página {objetivo} quedó en el puesto: {puesto}\n")print("""  Ningún buscador es perfecto en el puesto 1. Por eso RAG pasa VARIOS  fragmentos (k=3 o k=5) en vez de uno.  Pero ojo con el otro extremo: más contexto no siempre ayuda. Con k=10 el  modelo puede distraerse con nueve fragmentos irrelevantes y responder PEOR  que con k=1. No existe un k correcto universal: se calibra midiendo.""")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════#  CELDA 8 — GENERACIÓN con DeepSeek  (aquí sí se necesita la llave)##  DeepSeek es compatible con el SDK de OpenAI: solo cambia el base_url.# ═══════════════════════════════════════════════════════════════════════════from openai import OpenAIcliente_llm = OpenAI(api_key=API_KEY, base_url=CONFIG["base_url"]) if API_KEY else NoneSYSTEM_PROMPT = """Eres un asistente experto en el reglamento de Beca 18 de PRONABEC.REGLAS ESTRICTAS:1. Responde UNICAMENTE con informacion contenida en el CONTEXTO entregado.2. Cita siempre la pagina de la que sacaste el dato, en el formato (pag. N).3. Si el contexto no contiene la respuesta, responde exactamente:   "No encuentro eso en el reglamento que tengo cargado."   No completes con conocimiento general ni supongas.4. Responde en espanol claro, en maximo 5 oraciones."""SIN_RESPUESTA = "No encuentro eso en el reglamento que tengo cargado."LOG = []          # la contabilidad: una fila por llamadadef costo_usd(tin, tout, tcache=0):    """costo = (tokens_in/1e6)*precio_in + (tokens_out/1e6)*precio_out    Los tokens que entran por caché se cobran a una fracción del precio."""    p = CONFIG["precios"]    frescos = max(tin - tcache, 0)    return (frescos/1e6)*p["in"] + (tcache/1e6)*p["in_cache"] + (tout/1e6)*p["out"]def responder(pregunta, k=None, temperature=None):    """EL MOTOR. Recupera, decide si vale la pena preguntar, genera y registra."""    k = k or CONFIG["k"]    temperature = CONFIG["temperature"] if temperature is None else temperature    t0 = time.time()    fuentes = buscar(pregunta, k)    base = {"pregunta": pregunta, "fuentes": fuentes, "tokens_in": 0,            "tokens_cache": 0, "tokens_out": 0, "costo_usd": 0.0}    # CORTO CIRCUITO: si nada se parece lo suficiente, ni llamamos al modelo.    # Ahorra dinero y elimina la principal vía de invención.    mejor = fuentes[0]["similitud"] if fuentes else 0.0    if mejor < CONFIG["umbral_similitud"]:        return {**base, "respuesta": SIN_RESPUESTA, "abstuvo": True,                "motivo": f"similitud máxima {mejor} < umbral {CONFIG['umbral_similitud']}",                "latencia_s": round(time.time()-t0, 3)}    if cliente_llm is None:        return {**base, "respuesta": "(sin llave: no se llamó al modelo)",                "abstuvo": False, "motivo": "falta DEEPSEEK_API_KEY",                "latencia_s": round(time.time()-t0, 3)}    contexto = "\n\n".join(f"[Fragmento {i+1} | pagina {f['pagina']}]\n{f['texto']}"                           for i, f in enumerate(fuentes))    try:        r = cliente_llm.chat.completions.create(            model=CONFIG["modelo_llm"],            messages=[{"role": "system", "content": SYSTEM_PROMPT},                      {"role": "user",                       "content": f"CONTEXTO DEL DOCUMENTO:\n{contexto}\n\nPREGUNTA: {pregunta}"}],            temperature=temperature,        # 0.1: un chatbot normativo no puede variar            max_tokens=CONFIG["max_tokens"],  # techo SIEMPRE: sin techo no hay presupuesto        )        texto = r.choices[0].message.content        tin = getattr(r.usage, "prompt_tokens", 0)        tout = getattr(r.usage, "completion_tokens", 0)        tcache = getattr(r.usage, "prompt_cache_hit_tokens", 0) or 0        exito, err = True, ""    except Exception as e:        texto, tin, tout, tcache = f"Error: {e}", 0, 0, 0        exito, err = False, str(e)[:120]    lat = round(time.time()-t0, 3)    c = costo_usd(tin, tout, tcache)    LOG.append({"modelo": CONFIG["modelo_llm"], "tokens_in": tin,                "tokens_cache": tcache, "tokens_out": tout,                "costo_usd": c, "latencia_s": lat, "exito": exito, "error": err})    return {**base, "respuesta": texto,            "abstuvo": texto.strip().startswith(SIN_RESPUESTA[:20]),            "motivo": "", "tokens_in": tin, "tokens_cache": tcache,            "tokens_out": tout, "costo_usd": c, "latencia_s": lat}# --- Demo: una pregunta real ------------------------------------------------r = responder("¿Cuál es el monto de la subvención económica?")print("PREGUNTA:", r["pregunta"], "\n")print("RESPUESTA:")print(r["respuesta"], "\n")print(f"páginas citadas : {sorted({f['pagina'] for f in r['fuentes']})}")print(f"similitud top-1 : {r['fuentes'][0]['similitud']}")print(f"tokens          : {r['tokens_in']} entrada "      f"(+{r['tokens_cache']} de caché) / {r['tokens_out']} salida")print(f"costo           : USD {r['costo_usd']:.8f}")print(f"latencia        : {r['latencia_s']} s")# --- La comparación que justifica todo el pipeline -------------------------print(f"\nMandar el documento completo ({tokens_est:,} tokens) en cada pregunta:")print(f"  USD {costo_usd(tokens_est, 300):.6f}")print(f"RAG con k={CONFIG['k']} (~{r['tokens_in']} tokens):")print(f"  USD {costo_usd(r['tokens_in'], r['tokens_out']):.6f}")print(f"  -> {costo_usd(tokens_est,300)/max(costo_usd(r['tokens_in'],r['tokens_out']),1e-12):.0f} veces más barato")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════#  CELDA 9 — EVALUACIÓN: los DOS modos de falla de un RAG##  Recall@k            -> mide la BÚSQUEDA (chunking + embeddings). Gratis.#  Tasa de abstención  -> mide el PROMPT (que no invente). Consume API.##  Son cosas distintas y el arreglo es OPUESTO:#    falla de recuperación  -> el fragmento correcto no entró: mejor chunking, MÁS k#    falla de generación    -> entró pero el modelo se distrajo: menos ruido, MENOS k#  Subir k arregla una y empeora la otra.# ═══════════════════════════════════════════════════════════════════════════CASOS = [    # pregunta, tipo, páginas donde está la respuesta (verificadas a mano)    ("¿Cuál es el monto de la subvención económica de la beca?", "dominio", {9,13,31,38,92,93,111}),    ("¿Qué modalidades de beca existen en la convocatoria 2026?", "dominio", {3,13,14,59,60,89}),    ("¿Bajo qué causales se puede perder la beca?",              "dominio", {111}),    ("¿Cuáles son las obligaciones del becario?",                "dominio", {122,136}),    ("¿Me devuelven el costo de postulación?",                   "dominio", {39}),    ("¿En qué casos se suspende la beca?",                       "dominio", {136}),    ("¿Cuál es la receta del ceviche peruano?",                  "fuera",   set()),    ("¿Quién ganó el Mundial de 2022?",                          "fuera",   set()),    ("¿Cuál es la capital de Francia?",                          "fuera",   set()),]dominio = [c for c in CASOS if c[1] == "dominio"]fuera = [c for c in CASOS if c[1] == "fuera"]print("RECALL@k  —  mide la búsqueda, no cuesta nada\n")for k in (1, 3, 5):    aciertos = 0    detalle = []    for preg, _, esperadas in dominio:        recuperadas = {f["pagina"] for f in buscar(preg, k)}        ok = bool(esperadas & recuperadas)        aciertos += ok        detalle.append(("OK " if ok else "NO ", preg[:46], sorted(recuperadas)))    print(f"Recall@{k} = {aciertos/len(dominio):.2f}  ({aciertos}/{len(dominio)})")    for est, q, rec in detalle:        print(f"   {est} {q:<48} {rec}")    print()print("""  CÓMO SE LEE LA CURVA: si sube fuerte con k, el buscador encuentra la zona  correcta pero no la pone primera. Síntoma típico de un corpus con mucha  fórmula legal repetida ("de conformidad con", "según lo establecido en"):  esos fragmentos se parecen entre sí más de lo que se parecen a una pregunta.  DIAGNÓSTICO: el problema es la RECUPERACIÓN, no el LLM.  Cambiar de DeepSeek a otro modelo no movería estos números ni un punto.""")# --- Abstención: mide el prompt --------------------------------------------if cliente_llm:    print("ABSTENCIÓN  —  mide el prompt, sí consume API\n")    correctas = 0    for preg, _, _ in fuera:        rr = responder(preg)        ok = bool(rr["abstuvo"])        correctas += ok        print(f"   {'OK   ' if ok else 'FALLA'} {preg[:44]:<46} -> {rr['respuesta'][:52]}")    print(f"\nTasa de abstención correcta = {correctas/len(fuera):.2f} "          f"({correctas}/{len(fuera)})")    print("""  La última pregunta es deliberada: el modelo SÍ sabe que la capital de  Francia es París. Si la contesta, el system prompt no se está respetando,  y mañana va a inventar un monto de subvención.""")else:    print("(sin llave: se omite la prueba de abstención)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════#  CELDA 10 — BATERÍA FINAL Y CIERRE# ═══════════════════════════════════════════════════════════════════════════PREGUNTAS = [    "¿Qué modalidades de beca existen?",    "¿Bajo qué causales se puede perder la beca?",    "¿Cuáles son las obligaciones del becario?",    "¿Cuál es la capital de Francia?",        # debe abstenerse]for q in PREGUNTAS:    r = responder(q)    marca = "[SE ABSTUVO] " if r["abstuvo"] else ""    print(f"\nP: {q}")    print(f"R: {marca}{r['respuesta'][:300]}")    if not r["abstuvo"]:        print(f"   páginas {sorted({f['pagina'] for f in r['fuentes']})} | "              f"{r['tokens_in']}+{r['tokens_out']} tokens | "              f"USD {r['costo_usd']:.8f} | {r['latencia_s']}s")# --- La contabilidad de la sesión ------------------------------------------if LOG:    ok = [l for l in LOG if l["exito"]]    tin = sum(l["tokens_in"] for l in ok)    tca = sum(l["tokens_cache"] for l in ok)    tou = sum(l["tokens_out"] for l in ok)    tot = sum(l["costo_usd"] for l in ok)    lat = [l["latencia_s"] for l in ok]    print("\n" + "=" * 70)    print("CONTABILIDAD DE LA SESIÓN")    print(f"  llamadas exitosas : {len(ok)} de {len(LOG)}")    print(f"  tokens entrada    : {tin:,}  (de caché: {tca:,} = "          f"{tca/max(tin,1)*100:.0f}%)")    print(f"  tokens salida     : {tou:,}")    print(f"  costo total       : USD {tot:.6f}")    print(f"  costo por consulta: USD {tot/len(ok):.6f}")    print(f"  latencia media    : {sum(lat)/len(lat):.2f} s")    print(f"\n  Precios verificados el {CONFIG['precios']['verificado_el']}")    print(f"  Fuente: {CONFIG['precios']['fuente']}")    print("\n  Un costo que no está en el log NO va en el informe.")print("""======================================================================LAS SIETE COSAS QUE TE LLEVAS1. Mide antes de programar. El documento cabía en el modelo; lo que no   cabía era el presupuesto.2. Texto en el texto, datos sobre el texto en METADATA. El marcador   embebido dejaba al 91% de los fragmentos sin poder citar, y el modelo   citaba páginas equivocadas con total seguridad.3. El chunking es una decisión medible. Subir k para tapar una   recuperación mala es fuerza bruta.4. Lee la documentación de tu modelo de embeddings. Omitir los prefijos   no da error: solo empeora los resultados, en silencio.5. Distancia no es similitud. Y los umbrales no se trasladan entre modelos.6. Un RAG falla de dos maneras OPUESTAS. Si Recall@3 es bajo, el problema   es la búsqueda, no el LLM.7. Sin log de tokens y sin set de evaluación no tienes un producto:   tienes un demo.======================================================================TU PROYECTOAplica este mismo pipeline a un documento de tu dominio:  [ ] Diagrama Mermaid en el README, offline separado de online  [ ] config sin un solo número dentro del código  [ ] Página o sección como metadata, y cita en toda respuesta  [ ] Dos chunkings comparados CON Recall@k, no con argumentos  [ ] 30 preguntas de evaluación, 5 de ellas fuera de dominio  [ ] Log de costos con una fila por llamada  [ ] Streamlit local funcionandoLa pregunta de la sustentacion:  "Tu RAG, ¿falla por recuperación o por generación?   Muéstrame el número que lo prueba."Version completa con Streamlit y bot de Telegram:  github.com/patrickmelorr-lang/BECA-18-RAG======================================================================""")